# Lab 7: Inference and Global Climate Change
*Elements of Data Science* <br>
Welcome to Lab 7! This week we move from simulation to **statistical inference**: using data to decide whether a pattern could plausibly be due to chance. We will ask a big question — *has the Earth warmed since 1850?* — and answer it three ways: with a traditional *t*-test, with a simulation, and with a sign test on year-to-year changes. This material is covered in [Chapter 11: Testing Hypotheses](https://inferentialthinking.com/chapters/11/Testing_Hypotheses.html) and [Chapter 12: Comparing Two Samples](https://inferentialthinking.com/chapters/12/Comparing_Two_Samples.html) of the online <i>Inferential Thinking</i> textbook.
<br>**<center>Learning Goals**

|Area|Concept|
|---|---|
|Data cleaning|Remove non-country rows with `.where()` and `are.not_equal_to()`|
|Tables|`.join()`, `.relabel()`, `.pivot()`, `.group()` with a custom function|
|Hypotheses|State a hypothesis and a null hypothesis ([Ch. 11.1](https://inferentialthinking.com/chapters/11/1/Assessing_a_Model.html))|
|Traditional test|Unpaired vs. **paired** *t*-test: signal ÷ noise|
|Simulation test|Build the null distribution with `np.random.choice` and a `for` loop|
|p-values|Compute and interpret a p-value ([Ch. 11.3](https://inferentialthinking.com/chapters/11/3/Decisions_and_Uncertainty.html))|
|Sign test|Count increases minus decreases and compare to chance|
|Interpretation|What a small p-value does — and does **not** — tell us about causation|

First, set up the tests and imports by running the cells below.

## ⚙️ Initialization — DO NOT EDIT
The next two cells load the course tools (auto-checks, quizzes, styling) and record the lab version. **Do not change them.**

👉 **Run the cell below first** (Shift+Enter) before anything else in this lab.

In [ ]:
# 🔧 Run this cell first! (Shift+Enter)
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "shared-public")))
from init import *

In [ ]:
# 🔖 Lab version — do not edit. Used by the automated checks below.
ver = '2026V107'

### 🧰 Standard imports — start your own notebooks with these
The `init` cell above loads the course tools (auto-checks, quizzes, styling). The four lines below are the **only** imports you need for your own data science notebooks outside of this course — copy them into the first cell of any new notebook:

```python
import numpy as np                 # arrays, np.arange, np.random.choice, np.mean ...
from datascience import *          # Table, make_array, are ...
%matplotlib inline
import matplotlib.pyplot as plt    # plotting
```
Run them here too so every name is defined:

In [ ]:
# Standard imports for this lab
import numpy as np
from datascience import *
%matplotlib inline
import matplotlib.pyplot as plt

Now run the cell below to enter your name.

In [ ]:
# Enter your name as a string
name = ...

## 1. Background: Global Climate Change

*Climate change* usually refers to the long-term warming of global temperatures, together with shifts in weather patterns such as heat waves, heavy rainfall, and storms. Climate is the **long-term average** of weather (typically over 30 years or more), so a single cold winter says little about climate.

The climate can shift for natural reasons, but scientific assessments conclude that human activities — mainly burning fossil fuels, which raises atmospheric carbon dioxide and strengthens the greenhouse effect — are the dominant cause of warming since the mid-20th century. One alternative explanation, a brighter sun, does not fit the evidence: solar activity has not increased over the period when temperatures rose.

<img src='solar_temp.jpg' width="500" height="340">

**Read more:** [NASA — Causes of climate change](https://climate.nasa.gov/causes/) · [Government of Canada — Causes of climate change](https://www.canada.ca/en/environment-climate-change/services/climate-change/causes.html) · [IPCC AR6 Summary for Policymakers](https://www.ipcc.ch/report/ar6/wg1/chapter/summary-for-policymakers/)

*In this lab you will not simply take these conclusions on faith — you will test one piece of the evidence yourself.*

### Data
Many measurements could tell us about climate change; for simplicity we focus on **average annual land temperature by country**. The table was reshaped from Berkeley Earth's land temperatures by country ([Kaggle: Climate Change — Earth Surface Temperature Data](https://www.kaggle.com/datasets/berkeleyearth/climate-change-earth-surface-temperature-data), file `GlobalLandTemperaturesByCountry.csv`; see also [Berkeley Earth](https://berkeleyearth.org/data/)) to make our analysis easier. There are 15 columns: `year`, `country`, the average temperature for that year `avg` (°C), and one column for each month.

In [ ]:
temps = Table.read_table('temp_per_country.csv')
temps

### Data Exploration
Let's start by listing every unique entry in the `country` column.

In [ ]:
np.unique(temps.column('country'))

Notice that some entries are **continents**, not countries (e.g. *Africa*, *Europe*), and there are a few duplicates such as *Netherlands* and *Netherlands (Europe)*. Keeping them would count some land twice. Cleaning the data is virtually always the first step, so let's remove them.

In [ ]:
# Use the where() method in a loop to remove unwanted rows
non_countries_and_dups = make_array(
    "Africa",
    "Denmark (Europe)",
    "Europe",
    "France (Europe)",
    "French Southern And Antarctic Lands",
    "Kingman Reef",
    "Netherlands (Europe)",
    "North America",
    "Oceania",
)
for country in non_countries_and_dups:
    temps = temps.where('country', are.not_equal_to(country))

print('Number of countries after cleaning:', len(np.unique(temps.column('country'))))

### 📝 Quick Check — Tables and Climate Background
Test what you've learned about the climate background and the `Table` methods we just used. This is just for practice — answer as many times as you like, it isn't recorded.

In [ ]:
with open("questions07A.json", "r") as f:
    display_quiz(json.load(f), shuffle_answers=True)

## 2. Has the Earth Warmed Between 1850 and 2000?

If we look at the temperature change for every country between 1850 and 2000, what will we see? The cell below builds one row per country with its 1850 temperature (`T1`), its 2000 temperature (`T2`), and the change `T_diff = T2 - T1`. Only countries with data in **both** years survive the `.join()`.

In [ ]:
T_1850 = temps.where('year', 1850).select('year', 'country', 'avg')
T_2000 = temps.where('year', 2000).select('year', 'country', 'avg')

temp_change = T_1850.join('country', T_2000, 'country').relabel('avg', 'T1').relabel('avg_2', 'T2')
temp_change = temp_change.with_columns('T_diff', temp_change.column('T2') - temp_change.column('T1'))

print('Countries with data in both years:', temp_change.num_rows)
temp_change.show(3)

### <font color=blue> **Question 1.** </font>
Calculate the average temperature change (`T_diff`) across all countries in the data set.

In [ ]:
mean_diff = ...
mean_diff

In [ ]:
check('tests/q1.py')

### <font color=blue> **Question 2.** </font>
Make a histogram of the temperature changes.

In [ ]:
temp_change.hist('T_diff')

<font color=green>In the markdown cell below, describe the distribution shown in the histogram. Where is it centered? How many countries cooled? Do you think these changes could be random? Why or why not?</font>

---

ANSWER

---

#### Save notebook before check

In [ ]:
check('tests/q2.py')

### <font color=blue> **Question 3.** </font>
Let's test whether the change in mean temperature between 1850 and 2000 is *statistically significant*. In the cells below, state a hypothesis and a null hypothesis. Remember that the null hypothesis must describe a **chance model** we can simulate ([Ch. 11.1](https://inferentialthinking.com/chapters/11/1/Assessing_a_Model.html)).

#### <font color=green>Hypothesis:</font>

ANSWER

#### <font color=green>Null Hypothesis:</font>

ANSWER

#### Save notebook before check

In [ ]:
check('tests/q3.py')

## 3. Testing the Null Hypothesis Two Ways
We will test the null hypothesis two ways:
1. **The traditional way:** a *t*-test, as taught in most statistics classes. We'll see why the *paired* version is the right one for our data.
2. **The simulation way:** the approach used in this class from now on — no formulas or lookup tables, just code.

### 3.1 The *t*-test: signal versus noise
The *t*-test was developed by a chemist, William Gosset, working for the Guinness brewery around 1908. Guinness did not want competitors to learn that it used statistics for quality control, so he published under the pen name "Student" — hence *Student's t-test*. (Reference: [NIST — two-sample t-test](https://www.itl.nist.gov/div898/handbook/eda/section3/eda353.htm).)

Every *t*-test has the same idea:

$$ t = \frac{\text{signal}}{\text{noise}} = \frac{\text{difference in means}}{\text{standard error}} $$

The **standard error (SE)** of a mean is the sample standard deviation divided by the square root of the number of observations:

$$ SE = \frac{s}{\sqrt{n}} $$

It tells you how much the sample mean would vary if you repeated the study with new samples from the same population. More data → smaller SE → less uncertainty in the mean.

If the signal is large compared with the noise, the difference is hard to explain by chance and we reject the null hypothesis. If the signal is small compared with the noise, the two distributions blur together and chance remains a plausible explanation.

The key question is: **what counts as "noise"?** That depends on how the data were collected, which leads to two different versions of the test.

### 3.2 The two-sample (unpaired) *t*-test
The classic two-sample *t*-test compares **two independent groups** — for example, 50 patients who got a drug and a *different* 50 patients who got a placebo. Gosset showed that:
* **IF** the two samples are independent of each other,
* **and IF** the samples are random,
* **and IF** both populations are approximately normally distributed,
* **THEN** we can compute

$$ t = \frac{\bar{x}_2 - \bar{x}_1}{SE_{\text{combined}}}, \qquad SE_{\text{combined}} = \sqrt{SE_1^2 + SE_2^2} = \sqrt{\frac{s_1^2}{n_1} + \frac{s_2^2}{n_2}} $$

with $n_1 + n_2 - 2$ degrees of freedom. Here the "noise" is the spread of **each whole group**.

Let's see what happens if we (incorrectly) treat the 1850 temperatures and the 2000 temperatures as two unrelated groups. Run the cell below.

In [ ]:
# Two-sample (unpaired) t-test — treating 1850 and 2000 as unrelated groups
T1 = temp_change.column('T1')
T2 = temp_change.column('T2')
n = temp_change.num_rows

s1 = np.std(T1, ddof=1)          # ddof=1 gives the *sample* standard deviation
s2 = np.std(T2, ddof=1)
se_combined = np.sqrt(s1**2 / n + s2**2 / n)
t_unpaired = (np.mean(T2) - np.mean(T1)) / se_combined

print(f'Spread of 1850 temperatures across countries (s1): {s1:.2f} °C')
print(f'Spread of 2000 temperatures across countries (s2): {s2:.2f} °C')
print(f'Unpaired t = {t_unpaired:.2f} with {2*n - 2} degrees of freedom')

The unpaired *t* is small — below the critical value of 1.96 in the table further down — so this test would say the warming **could be due to chance**. Why? Countries range from the Sahara to Greenland, so temperatures *between* countries spread over tens of degrees. That huge country-to-country spread is treated as noise, and it drowns out a warming signal of about 1.5 °C.

But our two groups are **not** independent: they are the *same countries* measured twice. The unpaired test ignores that structure.

### 3.3 The paired *t*-test — the right test for our data
A **paired *t*-test** is used when each observation in one group is naturally matched to exactly one observation in the other: the same patient before and after treatment, the same field plot in two seasons, or — in our case — **the same country in 1850 and in 2000.**

Instead of comparing two group averages, we first compute the difference **within each pair** (`T_diff`), then ask whether the average difference is far from zero:

$$ t = \frac{\bar{x}_{\text{diff}}}{SE_{\text{diff}}}, \qquad SE_{\text{diff}} = \frac{s_{\text{diff}}}{\sqrt{n}}, \qquad \text{degrees of freedom} = n - 1 $$

where $n$ is the number of **pairs** (countries).

**Why pairing fits the warming question:**
* Each country serves as **its own control**. Subtracting Chad's 1850 temperature from Chad's 2000 temperature cancels out everything that makes Chad hot — latitude, elevation, geography — leaving only the change over time.
* The noise is now the spread of the *changes* (a fraction of a degree), not the spread of the climates (many degrees).
* The null hypothesis becomes: *the average within-country change is zero*.

**Assumptions to keep in mind:** the differences should be roughly normal (check your Question 2 histogram) and the pairs should be independent of each other. Is it realistic that neighboring countries warm independently? Keep that question in mind for the end of the lab.

The picture below shows why pairing matters: each line is one country. The countries are spread far apart vertically (big noise for an unpaired test), but nearly every line tilts upward (a consistent paired signal).

In [ ]:
# Each line connects one country's 1850 temperature to its 2000 temperature
for i in np.arange(temp_change.num_rows):
    plt.plot(make_array(1850, 2000), make_array(T1.item(i), T2.item(i)), color='gray', alpha=0.3)
plt.xticks(make_array(1850, 2000))
plt.ylabel('Average annual temperature [°C]')
plt.title('Same countries, measured twice')
plt.show()

In [ ]:
# Paired t-test ingredients
mean_diff = np.mean(temp_change.column('T_diff'))
s_diff = np.std(temp_change.column('T_diff'), ddof=1)
n = temp_change.num_rows
std_error = s_diff / np.sqrt(n)
dof = n - 1

print(f'The mean temperature change is: {mean_diff:.2f} °C')
print(f'The standard deviation of the temperature differences is: {s_diff:.3f} °C')
print(f'The standard error is: {std_error:.4f} °C')
print(f'The degrees of freedom (pairs - 1) is: {dof}')

### <font color=blue> **Question 4.** </font>
Calculate the **paired** *t*-value using `mean_diff` and `std_error`. Compare it with the unpaired value above.

In [ ]:
t = ...
print(f'The paired t value is: {t:.2f}')

In [ ]:
check('tests/q4.py')

### Find the p-value
This is where the "magic" happens in traditional statistics classes.

**Degrees of freedom:** the number of observations minus the number of quantities already estimated from the data. For a paired test we estimated one mean from $n$ differences, so the degrees of freedom are $n - 1$. For large data sets the degrees of freedom are nearly equal to the number of observations.

**Looking up p-values:** with a *t*-statistic and degrees of freedom, you can use [an online calculator](https://www.statology.org/t-score-p-value-calculator/) or a table of critical values like the one below. If your *t* is larger than the critical value in a column, your p-value is smaller than that column's p. For example, with 10 degrees of freedom and *t* = 2.30, we exceed 2.23 but not 3.17, so 0.01 < p < 0.05. For large degrees of freedom the *t*-distribution is nearly normal, so use the row labelled $\infty$.

**One- or two-tailed?** This table is *two-tailed*: it counts surprises in either direction. Our hypothesis has a direction (warming), so the one-tailed p-value is half the two-tailed value. Reading the two-tailed column is therefore the conservative choice.

<br>**<center>Critical Values of <i>t</i> (two-tailed)</center>**
<center>See: <a href="https://www.itl.nist.gov/div898/handbook/eda/section3/eda3672.htm">NIST</a></center>

|$\nu$<br>degrees of freedom|95%<br>p = 0.05|99%<br>p = 0.01|
|:-:|:--|:--|
|2|4.303|9.92|
|3|3.18|5.84|
|4|2.78|4.60|
|5|2.57|4.03|
|6|2.45|3.71|
|7|2.36|3.50|
|8|2.31|3.36|
|9|2.26|3.25|
|10|2.23|3.17|
|15|2.13|2.95|
|20|2.09|2.85|
|30|2.04|2.75|
|$\infty$|1.96|2.58|

### <font color=blue> **Question 5.** </font>
The **p-value** is the probability, *assuming the null hypothesis is true*, of getting a test statistic at least as extreme as the one we observed. Using your paired *t* and the table above, give an upper bound for the p-value (for example `0.05` or `0.01`).

In [ ]:
p = ...

In [ ]:
check('tests/q5.py')

### <font color=blue> **Question 5a.** </font>
<font color=green>Can we reject the null hypothesis? Why or why not? Would the *unpaired* test have led to the same conclusion? In one or two sentences, explain which test is appropriate here and why.</font>

---

ANSWER

---

#### Save notebook before check

In [ ]:
check('tests/q5a.py')

## 4. The Simulation Approach
Now we find a p-value the way we will for the rest of this course: no formulas, no tables, just simulation ([Ch. 11](https://inferentialthinking.com/chapters/11/Testing_Hypotheses.html)).

Under the null hypothesis there is no warming, so for each country **it would make no difference which of its two temperatures we label "1850" and which we label "2000."** This is just like a drug trial: if the drug does nothing, it doesn't matter which patients are labelled "treatment."

Swapping a country's two labels simply flips the sign of its difference: `T2 - T1` becomes `T1 - T2`. So to simulate one world with no warming we:
1. For each country, flip a fair coin: keep the difference (`+1`) or reverse it (`-1`).
2. Average the simulated differences across all countries.

Repeating this thousands of times builds the **distribution of mean differences we'd expect if there were no warming.** The p-value is the fraction of simulated means that are as large as or larger than the observed `mean_diff`.

The function below uses `np.random.choice` to flip that coin for one country.

In [ ]:
def shuffle_diff(T1, T2):
    '''
    Randomly assigns which temperature is "1850" and which is "2000".
    Returns either (T2 - T1) or (T1 - T2) with equal probability.
    '''
    sign = np.random.choice(make_array(1, -1))
    return sign * (T2 - T1)

In [ ]:
# Test the function: with T1 = 1 and T2 = 2 you should see a random mix of 1 and -1
for _ in np.arange(10):
    print(shuffle_diff(1, 2))

In [ ]:
# Apply the function to every row of our table
temp_change_simulate = temp_change.with_columns('T_diff_sim', temp_change.apply(shuffle_diff, 'T1', 'T2'))
temp_change_simulate.show(10)

### <font color=blue> **Question 5b.** </font>
<font color=green>Explain why, in the table above, the sign of `T_diff_sim` sometimes matches `T_diff` and sometimes is reversed, and why this is exactly what the null hypothesis predicts.</font>

---

ANSWER

---

#### Save notebook before check

In [ ]:
check('tests/q5b.py')

#### Simulate this over and over
`.apply()` flips one coin per row, which is clear but slow to repeat 10,000 times. The loop below does **exactly the same thing for all countries at once**: `np.random.choice` makes an array with one `+1` or `-1` per country, and multiplying it by the `T_diff` array flips the chosen differences.

In [ ]:
num_simulations = 10000
observed_diffs = temp_change.column('T_diff')
n = temp_change.num_rows

simulated_diffs = make_array()
for i in np.arange(num_simulations):
    signs = np.random.choice(make_array(1, -1), n)          # one coin flip per country
    simulated_diffs = np.append(simulated_diffs, np.mean(signs * observed_diffs))

simulated_diffs

### <font color=blue> **Question 6.** </font>
Make a histogram of `simulated_diffs`. *Hint:* use `plt.hist()` directly, or put the array into a Table with `Table().with_columns(...)` and use `.hist()`.

In [ ]:
...

### <font color=blue> **Question 7.** </font>
The simulated p-value is the fraction of simulated mean differences that are **greater than or equal to** the observed `mean_diff`. Calculate it. *Hint:* `np.count_nonzero()` counts the `True` values in a boolean array.

In [ ]:
p_val_simulation = ...
print(f"P-value from simulation: {p_val_simulation}")

In [ ]:
check('tests/q7.py')

### The null hypothesis is rejected
The histogram shows the mean differences we could get by chance if there were no warming. The triangle marks the **observed** mean change from 1850 to 2000. It lies far beyond every one of the 10,000 simulated values, so the simulated p-value is 0 — meaning p is smaller than about 1/10,000. We reject the null hypothesis: the warming between 1850 and 2000 is very unlikely to be due to chance.

In [ ]:
plt.hist(simulated_diffs, bins=20)
plt.plot(mean_diff, 0, '^', markersize=20, color='red')
plt.xlabel('Mean temperature difference [°C]')
plt.ylabel('Count')
plt.title('Null distribution (no warming) vs. observed change')
plt.show()

### <font color=blue> **Question 8.** </font>
**<font color=blue>Which approach do you prefer?</font>**

Would you rather learn statistical formulas, their assumptions, and rules for when each test applies — or learn a little Python and test hypotheses by simulation? Which do you prefer and why? We are really curious.

---

ANSWER

---
To see another example of hypothesis testing with simulation, read the [Deflategate chapter](https://inferentialthinking.com/chapters/12/3/Deflategate.html) of your textbook — it also compares paired measurements (the same footballs measured by two officials).

#### Save notebook before check

In [ ]:
check('tests/q8.py')

### 📝 Quick Check — Hypothesis Tests and *t*-Tests
Test what you've learned about null hypotheses, p-values, and paired versus unpaired *t*-tests. This is just for practice — answer as many times as you like, it isn't recorded.

In [ ]:
with open("questions07B.json", "r") as f:
    display_quiz(json.load(f), shuffle_answers=True)

## 5. Testing a Trend: The Sign Test

### Build a pivot table
The cell below creates a pivot table with one row per year and one column per country, filled with the average annual temperature `avg`.

⚠️ **Watch out:** when a country has no data for a year, `.pivot()` fills that cell with **0**. Temperatures of exactly 0 °C in those rows are *missing data*, not real measurements.

In [ ]:
pivotTable = temps.select('year', 'country', 'avg').pivot('country', 'year', 'avg', sum)
pivotTable

#### <font color=blue> Select two countries of your choice to study </font>
Select two countries and draw a line plot of their temperatures over time. Only graph years with data for **both** countries (*Hint:* use `.select()` to choose the columns and `.where('year', are.above(...))` to drop early years — then check that no zeros remain). There is no auto-check, since everyone may choose different countries.

In [ ]:
yourCountries = pivotTable.select('year', ..., ...).where('year', are.above(1850))
yourCountries.show(5)

In [ ]:
yourCountries.plot('year')
plt.ylabel('Temperature [°C]')
plt.xticks(np.arange(1850, 2025, 25))
plt.show()

### <font color=blue> **Question 9.** </font>
<font color=green>Describe one observation from the figure you generated. Is there a trend? How large are the year-to-year wiggles compared with the long-term change?</font>

---

ANSWER

---

#### Save notebook before check

In [ ]:
check('tests/q9.py')

### <font color=blue> **Question 10.** </font>
This time we look at the **overall trend** rather than only the start and end temperatures. Based on our figures and what we know about good hypotheses, write the null hypothesis in the cell below.

- **Hypothesis:** Country temperatures tend to trend upward over time.
- **Null hypothesis:** *(your answer in the next cell)*

---

ANSWER

---

#### Save notebook before check

In [ ]:
check('tests/q10.py')

Temperatures vary widely across countries and years. Rather than analyzing the temperatures themselves, we restrict our analysis to the **direction** of change: over each two-year span, did the temperature go up or down? We ignore *how much* it changed. This is called a **sign test**.

`np.diff` computes differences between adjacent items:

    [item 1 - item 0 , item 2 - item 1 , item 3 - item 2, ...]

We want differences between items **two positions apart**:

    [item 2 - item 0 , item 3 - item 1 , item 4 - item 2]

The `diff_n` function below computes this result.

In [ ]:
def diff_n(values, n):
    '''
    Parameters:
    values is an array of numbers
    n is the offset (how far apart the numbers are in the array)

    Example:
    If values = [2, 6, 8, 9, 15] and n = 2,
    the function subtracts values that are 2 apart: (8 - 2), (9 - 6) and (15 - 8)
    '''
    return np.array(values)[n:] - np.array(values)[:-n]

diff_n(make_array(2, 6, 8, 9, 15), 2)

### <font color=blue> **Question 11.** </font>
Implement the function `changes`, which takes an array of temperatures for one country ordered by increasing year. For all two-year periods (e.g. 1960 to 1962) it returns **the number of increases minus the number of decreases.**

For example, `r = make_array(10, 7, 12, 9, 13, 9, 11)` has three increases (10→12, 7→9, 12→13), one decrease (13→11), and one change that is neither (9→9). So `changes(r)` returns `3 - 1 = 2`.

*Hint:* use `diff_n`, then count the `True` values of a comparison with `np.count_nonzero` — once for increases and once for decreases. Python counts `True` as 1 and `False` as 0.

Use the `years` argument instead of typing `2`, so the same function works for any spacing. The auto-check also tries `years=5`.

In [ ]:
def changes(array, years=2):
    "Return the number of increases minus the number of decreases over `years`-year periods."
    ...

In [ ]:
print(changes(make_array(10, 7, 12, 9, 13, 9, 11)))      # should be 2
print(changes(make_array(10, 7, 12, 9, 13, 9, 11), 4))   # should be 1

In [ ]:
check('tests/q11.py')

### <font color=blue> **Question 12.** </font>
Assign `changes_by_country` to a table with one row per country and two columns: `country` and `avg changes` (the `changes` statistic for that country's `avg` temperatures across all years). The first rows should look like this:

|country    |avg changes|
|-----------|-----------|
|Afghanistan|18         |
|Albania    |-22        |

**Hint:** `.group('country', changes)` applies your function to *every other column* for each country. Use `.select()` first so only the column you need is left. Here is an example with Winter Olympics data:

In [ ]:
NORUSA = Table.read_table('NORUSA.csv')
NORUSA_NUMBERS = NORUSA.group(['Year', 'Team'])   # Number of athletes per year
NORUSA_NUMBERS

Now compute increases minus decreases in team size for each team, grouping by `Team` across all Olympic years:

|Team| Year changes | count changes|
|----|---|---|
|Norway|20|10|
|United States|20|18|

In [ ]:
NORUSA_NUMBERS.group('Team', changes)

#### Now apply this to the temperature data
Start from the cleaned `temps` table (one row per country per year, already in year order) — not `temp_change`, which only has 1850 and 2000. As the `Year changes` column above shows, `.group()` applies `changes` to every column, so first keep only the columns you need. Assign the result to `changes_by_country`.

In [ ]:
changes_by_country = ...
changes_by_country

In [ ]:
check('tests/q12.py')

### <font color=blue> **Question 13.** </font>
Assign `test_stat` to the **total** increases minus total decreases over all two-year periods and all countries. For example, if Albania's temperature went up 23 times and fell 17 times, Albania contributes 6. Add up the contributions from every country.

In [ ]:
test_stat = ...
print('Total increases minus total decreases, across all countries and years:', test_stat)

In [ ]:
check('tests/q13.py')

"More increases than decreases," one person exclaims. "Temperatures tend to go up across two-year periods. What dire times we live in."

"Not so fast," another replies. "Even if temperatures moved up and down at random, there would be *some* difference between increases and decreases. There were many countries and many years, so many chances for change. If temperatures rise and fall with equal probability, perhaps this difference is simply due to chance!"

Under the null hypothesis, each two-year change is equally likely to be an increase or a decrease — like a fair coin. Our test statistic depends only on the direction of change:

    Test Statistic: The number of increases minus the number of decreases

The cell below samples 100 increases/decreases at random from a uniform distribution. The last column gives the counts. `sample_from_distribution` is faster than `.sample()` followed by `.group()`.

In [ ]:
uniform = Table().with_columns(
    "Change", make_array('Increase', 'Decrease'),
    "Chance", make_array(0.5,        0.5))
uniform.sample_from_distribution('Chance', 100)

### <font color=blue> **Question 14.** </font>
Under the null hypothesis, every two-year change is a fair coin flip: increase or decrease, 50/50. You'll build that chance model in three steps.

**Step 1 — How many coin flips?** A country with *k* years of data has *k* − 2 two-year comparisons. `temps.group('country')` gives each country's number of years in its `count` column. Subtract 2 from each count and add them up. (You should get 42,122.)

In [ ]:
# Step 1: total number of two-year comparisons across all countries
num_changes = ...
num_changes

**Step 2 — One simulated world.** Complete `simulate_under_null(num_chances_to_change)`. It should flip `num_chances_to_change` fair coins and return *increases minus decreases*. Either approach works:
- **The coin from Part 4:** `np.random.choice(make_array(1, -1), num_chances_to_change)` gives +1 for each increase and −1 for each decrease. What does adding them up give you?
- **The `uniform` table above:** `uniform.sample_from_distribution('Chance', num_chances_to_change)` adds a column `Chance sample`. Row 0 is the number of increases and row 1 the number of decreases. Get them with `.column('Chance sample').item(0)` and `.item(1)`.

In [ ]:
def simulate_under_null(num_chances_to_change):
    """Simulates num_chances_to_change changes, each equally likely to be
    an increase or a decrease. Returns the test statistic
    (increases minus decreases) for these simulated changes.
    """
    uniform = Table().with_columns(
        "Change", make_array('Increase', 'Decrease'),
        "Chance", make_array(0.5,        0.5))
    # Flip num_chances_to_change fair coins (either approach from Step 2)
    # and return increases minus decreases.
    ...

    return ...

In [ ]:
# Test it: a small even number near 0. Run this cell a few times.
simulate_under_null(100)

**Step 3 — Repeat 10,000 times.** In `empirical_distribution(tbl)`, set `num_changes` using your Step 1 calculation, with `tbl` in place of `temps`, and pass it to `simulate_under_null`. Then run `empirical_distribution(temps)`. The red triangle marks the observed `test_stat`.

In [ ]:
def empirical_distribution(tbl):
    num_changes = ...
    samples = make_array()
    for i in np.arange(10000):
        samples = np.append(samples, simulate_under_null(...))
    # Bins are centered on 0, the value the null hypothesis expects
    limit = max(abs(samples)) * 1.1
    Table().with_columns('Test statistic under null', samples).hist(bins=np.arange(-limit, limit, limit / 20))
    return samples

In [ ]:
samples = empirical_distribution(...)   # which table holds every country's yearly temperatures?
plt.plot(test_stat, 0, '^', markersize=20, color='red')   # observed test statistic
plt.show()

In [ ]:
check('tests/q14.py')

### <font color=blue> **Question 15.** </font>
Compute the p-value: the fraction of simulated test statistics that are **greater than or equal to** `test_stat`. (*Hint:* `np.count_nonzero()`.)

In [ ]:
pvalue = ...
pvalue

In [ ]:
check('tests/q15.py')

### <font color=blue> **Question 16. Discussion** </font>
<font color=green>
1. Using a 5% p-value cutoff, what do you conclude about the null and alternative hypotheses?
2. Describe your findings in simple, non-technical language. What does this analysis tell you about temperature changes over time?
3. What can — and can't — you claim about the **cause** of these changes from this statistical analysis?
</font>

---

ANSWER

---

#### Save notebook before check

In [ ]:
check('tests/q16.py')

### <font color=blue> **Question 17.** </font>
#### Summary
You have tested for global warming in two ways:

1. You compared each country's temperature in **1850** with its temperature in **2000** (paired *t*-test and simulation). The observed increase was too large to be explained by chance.
2. You looked at every two-year change in each country's full temperature record and found far more increases than decreases — again, too many to be explained by random fluctuation.

In both cases you rejected the null hypothesis. However, our toolbox for detecting trends is still limited, and climate scientists use more sophisticated methods that make better use of all the data.

<font color=green>If our goal is to detect trends in global temperatures, what limitations of our **methods** and our **data set** can you identify? Think about the assumptions behind each test, how the data were collected, and how the countries relate to each other.</font>

---

ANSWER

---

#### Save notebook before check

In [ ]:
check('tests/q17.py')

## 📝 Quick Review Quiz
Unlike the quick checks above, this quiz **is recorded** — you get two attempts per question, so take your time.

In [ ]:
# Interactive quiz — run this cell and answer the questions!
from stemds_quiz import open_quiz
quiz_result = open_quiz("questions07C.json", name, user, quiz_id="Lab07C", max_attempts=2)

## 6. Reflection

### <font color=blue> **Question 18.** </font>
At the end of each lab, please include a reflection.
* How did this lab go?
* Have you used a *t*-test before in another class? Do you now understand the difference between a paired and an unpaired test?
* Were there questions you found especially challenging that you would like your instructor to review in class?
* How long did the lab take you to complete?

Share your feedback so we can continue to improve this class!

**Use the markdown cell below this one and write your reflection on this lab.**

---

ANSWER

---

#### Save notebook before check

In [ ]:
check('tests/q18.py')

## 🏁 Final Step — Check, Submit to Canvas, Then Submit Lab

Follow these three steps **in order**:

1. ✅ **Run the cell below** to test all your answers and see your score.
2. 📤 **Upload to Canvas.** Download your notebook as a `.ipynb` file ([instructions](https://laserchemist.github.io/data/submission-instructions.html#ipynb)) and export an HTML copy ([instructions](https://laserchemist.github.io/data/submission-instructions.html#html)), then upload that HTML file to this lab's Canvas assignment.
3. 🚀 **Only then run the final cell and click Submit Lab.** This records your score to the course gradebook — treat it as confirmation that steps 1 and 2 are already done.

In [ ]:
# Run all tests at once
from gofer.ok import check
correct = 0
questions = ['1',
             '2',
             '3',
             '4',
             '5',
             '5a',
             '5b',
             '7',
             '8',
             '9',
             '10',
             '11',
             '12',
             '13',
             '14',
             '15',
             '16',
             '17',
             '18']
for x in questions:
    print("Testing question {}: ".format(x))
    display(check("tests/q{}.py".format(x)))
    score = check('tests/q{}.py'.format(str(x)))
    if score.grade == 1.0:
        correct += 1

In [ ]:
from lab_submit import submit_lab
submit_lab(name, user, correct, questions, lab_id="Lab07")